# 07 · ADF break sensitivity (research IS only)

Other test — **not** a STAR change / not a reopen of H-003. Compares standard break arms
plus notebook-local variants that keep the entry block but change flatten behaviour
(`no_flat`, flatten only when `pos * z` has already mean-reverted).

Self-contained panel preprocessing — engine / STAR / ledger untouched.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import enrich_trades, extreme_trades
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import cvar, fit_mean_abs_score
from strategies.s2_coint.engine import simulate_book
from strategies.s2_coint.metrics import corr_to_s1, metrics_from_returns_inference
from strategies.s2_coint.sizing import pair_scale_from_score

warnings.filterwarnings("ignore", category=FutureWarning)

N_EXTREME = 7
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("EXIT_STAR", stack.get("EXIT_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("EXIT_STAR", stack.get("EXIT_STAR"))
print("BREAK_STAR", stack.get("BREAK_STAR"))
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("TREND_STAR", stack.get("TREND_STAR"))
print("VOL_STAR", stack.get("VOL_STAR"))
print("NOTE: STAR stack is read-only in this notebook (other_tests).")

In [ ]:
def _cagr(returns: pd.Series, periods_per_year: float = 252.0) -> float:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    if r.empty:
        return float("nan")
    total = float((1.0 + r).prod())
    years = len(r) / float(periods_per_year)
    if years <= 0 or total <= 0:
        return float("nan")
    return float(total ** (1.0 / years) - 1.0)


def arm_metrics(returns: pd.Series, s1: pd.Series | None = None) -> dict:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    r.index = pd.to_datetime(r.index)
    m = metrics_from_returns_inference(r, periods_per_year=252.0)
    cagr = _cagr(r)
    mdd = float(m.get("max_drawdown", float("nan")))
    calmar = float(cagr / abs(mdd)) if np.isfinite(cagr) and np.isfinite(mdd) and mdd != 0 else float("nan")
    return {
        "ann_sharpe": m.get("ann_sharpe", float("nan")),
        "max_drawdown": mdd,
        "calmar": calmar,
        "cagr": cagr,
        "skew": m.get("skew", float("nan")),
        "excess_kurtosis": m.get("excess_kurtosis", float("nan")),
        "cvar_5pct": cvar(r, alpha=0.05),
        "corr_to_s1": corr_to_s1(r, s1 if s1 is not None else s1_weekly),
        "n_days": m.get("n_days", 0),
    }


def collect_trades(book, panel: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for pid, res in book.pair_results.items():
        if res.trades is None or res.trades.empty:
            continue
        t = res.trades.copy()
        t["pair_id"] = str(pid)
        frames.append(enrich_trades(t, panel, res.returns))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def show_extreme(trades: pd.DataFrame, n: int = N_EXTREME, title: str = "") -> None:
    best, worst = extreme_trades(trades, n=n)
    cols = [
        "pair_id", "side_label", "entry_date", "exit_date", "hold_bars",
        "exit_reason", "pnl_pct", "z_entry", "z_exit", "adf_entry", "adf_exit",
    ]
    if title:
        print(title)
    print(f"=== Top {n} trades ===")
    display(best[cols] if not best.empty else best)
    print(f"=== Bottom {n} trades ===")
    display(worst[cols] if not worst.empty else worst)


def metrics_table(rows: dict[str, dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).T
    order = [
        "ann_sharpe", "max_drawdown", "calmar", "cagr", "skew",
        "excess_kurtosis", "cvar_5pct", "corr_to_s1", "n_days",
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols]

from dataclasses import replace

In [ ]:
bar = str(stack.get("BAR_STAR") or "1d")
lb = lookbacks_for_bar(bar)
# Prefer cached STAR panels when present; else overlay hedge on the research-IS train panel only.
try:
    train_star, _full_star, _manifest = load_star_panels(
        universe=UNIVERSE, bar=bar, pair_ids=PAIRS, root=ROOT
    )
    is_end = is_end_for_stack(stack, train_star)
    is_raw, _oos_unused = split_is_oos(train_star, is_end=is_end)
    del _oos_unused
    panel_src = "cached star train"
except (FileNotFoundError, ValueError) as exc:
    print("star panels unavailable (", type(exc).__name__, ") — using universe train panel")
    train, _full = load_universe_panels(UNIVERSE, bar, PAIRS, root=ROOT)
    is_end = is_end_for_stack(stack, train)
    is_raw, _oos_unused = split_is_oos(train, is_end=is_end)
    del _oos_unused
    panel_src = "universe train"

hedge = str(stack.get("HEDGE_STAR") or "ols")
# Skip re-overlay when star cache already has z / adf columns.
need_overlay = "z" not in is_raw.columns or "adf_pvalue" not in is_raw.columns
if need_overlay:
    if hedge == "kalman":
        is_panel = overlay_kalman_hedge(
            is_raw,
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
    else:
        is_panel = overlay_ols_hedge(
            is_raw,
            ols_window=lb["ols_window"],
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
else:
    is_panel = is_raw.copy()

is_panel = is_panel.loc[is_panel["pair_id"].astype(str).isin(PAIRS)].copy()
is_panel["date"] = pd.to_datetime(is_panel["date"])

mean_abs = fit_mean_abs_score(is_panel, score_column="z")
s1_weekly = load_s1_weekly(ROOT)
cfg_star = config_from_stack(stack)
print("panel_src", panel_src, "need_overlay", need_overlay)
print("bar", bar, "is_end", is_end)
print("IS rows", len(is_panel), "pairs", is_panel["pair_id"].nunique())
print("frozen IS mean(|z|)", round(mean_abs, 4))
print("cfg", cfg_star)

## 1. Standard Break Arms

In [ ]:
results = {}
books = {}
trades_by = {}

def run_arm(name: str, cfg, panel) -> None:
    book = simulate_book(panel, cfg, mean_abs_score=mean_abs)
    books[name] = book
    results[name] = arm_metrics(book.returns)
    trades_by[name] = collect_trades(book, panel)
    t = trades_by[name]
    reasons = t["exit_reason"].value_counts().to_dict() if not t.empty else {}
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in results[name].items()}, reasons)

for mode in ("off", "block_05_flat_10", "flat_05"):
    run_arm(mode, replace(cfg_star, break_mode=mode), is_panel)

display(metrics_table({k: results[k] for k in ("off", "block_05_flat_10", "flat_05")}))

## 2. Novel Break Arms (notebook-local)

- **block_05_no_flat** — `break_mode="off"` (no health flatten) plus ADF>=0.05 **entry demotion masks** on `simulate_pair` (open trades run to z-exit).
- **block_05_flat_z0 / z02** — keep STAR entry block via `block_05_flat_10`, but only allow flatten when ADF/VJ would fire **and** `pos * z >= thresh` (two-pass panel restore using the no_flat position path).


In [ ]:
import strategies.s2_coint.engine as s2_engine
from risk.analytics.s1_equities.vol_targeting import VolTargetConfig
from strategies.s2_coint.engine import BookSimResult, simulate_pair


def simulate_book_adf_entry_block(
    panel: pd.DataFrame,
    cfg,
    *,
    mean_abs_score: float,
    block_adf: float = 0.05,
):
    """``break_mode=off`` book: block new entries when ADF >= block_adf via entry masks.

    STAR pairs share no legs, so independent ``simulate_pair`` + equal-weight mean
    matches ``never_allow`` for this book. VT overlay uses the engine helper.
    """
    cfg = replace(cfg, break_mode="off")
    pairs = list(panel.groupby("pair_id", sort=False))
    n_pairs = max(len(pairs), 1)
    parts: list[pd.Series] = []
    pair_results = {}
    for pid, g in pairs:
        g = g.sort_values("date").reset_index(drop=True)
        adf = (
            pd.to_numeric(g["adf_pvalue"], errors="coerce").to_numpy(dtype=float)
            if "adf_pvalue" in g.columns
            else np.full(len(g), np.nan)
        )
        allowed = ~(np.isfinite(adf) & (adf >= float(block_adf)))
        res = simulate_pair(
            g,
            cfg,
            mean_abs_score=mean_abs_score,
            n_pairs=n_pairs,
            long_entry_allowed=allowed,
            short_entry_allowed=allowed,
        )
        pair_results[str(pid)] = res
        if not res.returns.empty:
            parts.append(res.returns.rename(str(pid)))
    if not parts:
        empty = pd.Series(dtype=float, name="ret")
        return BookSimResult(returns=empty, returns_base=empty, pair_results=pair_results)
    wide = pd.concat(parts, axis=1).fillna(0.0)
    raw = wide.mean(axis=1).rename("ret")
    vt_cfg = VolTargetConfig(
        enabled=cfg.vol_mode == "s1_vt",
        target_ann_vol=cfg.vt_target_ann_vol,
        periods_per_year=252.0,
        min_periods=max(13, int(cfg.sigma_window // 4)),
    )
    return s2_engine._finalize_book(
        raw,
        panel,
        pair_results=pair_results,
        vt_cfg=vt_cfg if cfg.vol_mode == "s1_vt" else None,
    )


def _open_side_path(trades: pd.DataFrame, dates: pd.DatetimeIndex) -> pd.Series:
    """Map calendar dates -> open side (+1/-1/0) from a trade blotter."""
    side = pd.Series(0, index=pd.DatetimeIndex(dates), dtype=int)
    if trades is None or trades.empty:
        return side
    for row in trades.itertuples(index=False):
        a = pd.Timestamp(row.entry_date)
        b = pd.Timestamp(row.exit_date)
        s = int(row.side)
        mask = (side.index >= a) & (side.index < b)
        side.loc[mask] = s
    return side


def panel_block_flat_z(panel: pd.DataFrame, trades_nofloat: pd.DataFrame, thresh: float) -> pd.DataFrame:
    """Suppress flatten unless open pos*z >= thresh; entry block stays via break_mode."""
    raw = panel.copy()
    out = panel.copy()
    adf_all = pd.to_numeric(out["adf_pvalue"], errors="coerce").astype(float)
    out["adf_pvalue"] = adf_all.clip(upper=0.099)
    if "variance_jump" in out.columns:
        out["variance_jump"] = 0.0
    for pid, g in raw.groupby("pair_id", sort=False):
        g = g.sort_values("date")
        dates = pd.DatetimeIndex(pd.to_datetime(g["date"]))
        tr = (
            trades_nofloat.loc[trades_nofloat["pair_id"].astype(str) == str(pid)]
            if not trades_nofloat.empty
            else trades_nofloat
        )
        side_path = _open_side_path(tr, dates)
        z = pd.to_numeric(g["z"], errors="coerce").astype(float).to_numpy()
        adf = pd.to_numeric(g["adf_pvalue"], errors="coerce").astype(float).to_numpy()
        vj = (
            pd.to_numeric(g["variance_jump"], errors="coerce").astype(float).to_numpy()
            if "variance_jump" in g.columns
            else np.zeros(len(g))
        )
        sides = side_path.reindex(dates).fillna(0).astype(int).to_numpy()
        allow = (sides != 0) & np.isfinite(z) & ((sides.astype(float) * z) >= float(thresh))
        health = allow & (
            ((np.isfinite(adf) & (adf >= 0.10)) | (np.isfinite(vj) & (vj >= 2.0)))
        )
        idx = g.index.to_numpy()
        out.loc[idx[health], "adf_pvalue"] = adf[health]
        if "variance_jump" in out.columns:
            out.loc[idx[health], "variance_jump"] = vj[health]
    return out


# --- no_flat via demotion masks ---
cfg_off = replace(cfg_star, break_mode="off")
book_nf = simulate_book_adf_entry_block(
    is_panel, cfg_off, mean_abs_score=mean_abs, block_adf=0.05
)
books["block_05_no_flat"] = book_nf
results["block_05_no_flat"] = arm_metrics(book_nf.returns)
trades_by["block_05_no_flat"] = collect_trades(book_nf, is_panel)
t = trades_by["block_05_no_flat"]
reasons = t["exit_reason"].value_counts().to_dict() if not t.empty else {}
print(
    "block_05_no_flat",
    {k: round(v, 4) if isinstance(v, float) else v for k, v in results["block_05_no_flat"].items()},
    reasons,
)

nofloat_blotter_frames = []
for pid, res in book_nf.pair_results.items():
    if res.trades is None or res.trades.empty:
        continue
    tt = res.trades.copy()
    tt["pair_id"] = str(pid)
    nofloat_blotter_frames.append(tt)
nofloat_blotter = (
    pd.concat(nofloat_blotter_frames, ignore_index=True)
    if nofloat_blotter_frames
    else pd.DataFrame()
)

cfg_block = replace(cfg_star, break_mode="block_05_flat_10")
for thresh, name in [(0.0, "block_05_flat_z0"), (0.2, "block_05_flat_z02")]:
    panel_z = panel_block_flat_z(is_panel, nofloat_blotter, thresh)
    run_arm(name, cfg_block, panel_z)

display(
    metrics_table(
        {
            k: results[k]
            for k in ("block_05_no_flat", "block_05_flat_z0", "block_05_flat_z02")
        }
    )
)


## 3. Counterfactual Analysis (coint_break vs let-run)

In [ ]:
star_trades = trades_by["block_05_flat_10"]
nf_trades = trades_by["block_05_no_flat"]
cb = star_trades.loc[star_trades["exit_reason"] == "coint_break"].copy()
print("coint_break trades under STAR break:", len(cb))

# Match by pair_id + entry_date to no_flat outcome
nf_key = nf_trades.set_index(["pair_id", "entry_date"], drop=False)
rows = []
for row in cb.itertuples(index=False):
    key = (str(row.pair_id), pd.Timestamp(row.entry_date))
    alt = None
    if key in nf_key.index:
        hit = nf_key.loc[key]
        alt = hit.iloc[0] if isinstance(hit, pd.DataFrame) else hit
    rows.append({
        "pair_id": row.pair_id,
        "entry_date": row.entry_date,
        "exit_date_break": row.exit_date,
        "pnl_break": float(row.pnl_pct) if pd.notna(row.pnl_pct) else float("nan"),
        "z_exit_break": float(row.z_exit) if pd.notna(row.z_exit) else float("nan"),
        "adf_exit_break": float(row.adf_exit) if pd.notna(row.adf_exit) else float("nan"),
        "exit_date_nofloat": getattr(alt, "exit_date", pd.NaT) if alt is not None else pd.NaT,
        "pnl_nofloat": float(getattr(alt, "pnl_pct", float("nan"))) if alt is not None else float("nan"),
        "exit_reason_nofloat": getattr(alt, "exit_reason", "") if alt is not None else "",
    })
cf = pd.DataFrame(rows)
if not cf.empty:
    cf["delta_pnl"] = cf["pnl_nofloat"] - cf["pnl_break"]
    cf["break_helped"] = cf["delta_pnl"] < 0  # letting run was worse → break helped
    print("break helped (let-run worse):", int(cf["break_helped"].sum()), "/", len(cf))
    print("break hurt (let-run better):", int((~cf["break_helped"]).sum()), "/", len(cf))
    print("mean delta_pnl (nofloat - break):", float(cf["delta_pnl"].mean()))
    display(cf.sort_values("delta_pnl"))
else:
    print("No coint_break trades to compare.")

## 4. Comparison

In [ ]:
all_tbl = metrics_table(results)
display(all_tbl.sort_values("ann_sharpe", ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col, title in zip(
    axes,
    ["ann_sharpe", "excess_kurtosis", "cvar_5pct"],
    ["Sharpe", "Excess kurtosis", "CVaR 5%"],
):
    all_tbl[col].plot(kind="bar", ax=ax, color="#1f4e79", title=title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for arm in all_tbl.sort_values("excess_kurtosis").index[:2]:
    show_extreme(trades_by[str(arm)], title=f"Extremes: {arm}")

## 5. Summary

Are ADF flattens net-positive or net-negative on IS?
Does `block_05_no_flat` or a z-gated flatten improve kurtosis / Calmar vs STAR `block_05_flat_10`?
Promote to a new hypothesis later only if the effect is material.